# 6. Figures

## Import Packages

In [ ]:
import sys

sys.path.append("../../")
sys.path.append('../../src')

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import random
import matplotlib.pyplot as plt

from m3util.ml.rand import set_seeds
from m3util.viz.style import set_style
from m3util.viz.printing import printer
from belearn.viz.viz import Viz
from belearn.dataset.dataset import BE_Dataset
from belearn.functions.sho import SHO_nn

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

from m3util.viz.layout import inset_connector, add_box
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front

from matplotlib.gridspec import GridSpec
import matplotlib.image as mpimg

printing = printer(basepath = './Figures/')


set_style("printing")
set_seeds(seed=42)

%matplotlib inline

In [ ]:
def SHO_fit_func_nn(params,
                    wvec_freq,
                    device='cpu'):
    """_summary_

    Returns:
        _type_: _description_
    """

    Amp = params[:, 0].type(torch.complex128)
    w_0 = params[:, 1].type(torch.complex128)
    Q = params[:, 2].type(torch.complex128)
    phi = params[:, 3].type(torch.complex128)
    wvec_freq = torch.tensor(wvec_freq)

    Amp = torch.unsqueeze(Amp, 1)
    w_0 = torch.unsqueeze(w_0, 1)
    phi = torch.unsqueeze(phi, 1)
    Q = torch.unsqueeze(Q, 1)

    wvec_freq = wvec_freq.to(device)

    numer = Amp * torch.exp((1.j) * phi) * torch.square(w_0)
    den_1 = torch.square(wvec_freq)
    den_2 = (1.j) * wvec_freq.to(device) * w_0 / Q
    den_3 = torch.square(w_0)

    den = den_1 - den_2 - den_3

    func = numer / den

    return func

## Loads Data

In [ ]:
# Specify the filename and the path to save the file
filename = "data_raw.h5"
save_path = "./Data"

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path, SHO_fit_func_LSQF=SHO_fit_func_nn)

# print the contents of the file
dataset.print_be_tree()

In [ ]:
dataset.SHO_Scaler()

## Figure 1

In [ ]:
# instantiates the visualization object
BE_viz = Viz(dataset, printing, verbose=True)

In [ ]:
fig = BE_viz.raw_be(dataset, filename="Figure_2_raw_be_experiment")

In [ ]:
axes = fig.axes
axes

In [ ]:
def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    target_ax.set_xlabel(source_ax.get_xlabel())
    target_ax.set_ylabel(source_ax.get_ylabel())
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
        
    for line in source_ax.get_lines():
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color())
        
    if secondary_ax != None:
        if type_ax == 'twin':
            ax_twin = target_ax.twinx()
            for line in secondary_ax.get_lines():
                ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color())
        elif type_ax == 'same':
            for line in secondary_ax.get_lines():
                target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color())
                
                max_x_lim = max(source_ax.get_xlim()[1], secondary_ax.get_xlim()[1])
                min_x_lim = min(source_ax.get_xlim()[0], secondary_ax.get_xlim()[0])
                max_y_lim = max(source_ax.get_ylim()[1], secondary_ax.get_ylim()[1])
                min_y_lim = min(source_ax.get_ylim()[0], secondary_ax.get_ylim()[0])
                target_ax.set_xlim((min_x_lim, max_x_lim))
                target_ax.set_ylim((min_y_lim, max_y_lim))
        else:
            ax_new = target_ax.inset_axes([0.5, 0.65, 0.48, 0.33])
            x_start = 120
            x_end = 140
            ax_new.plot(dataset.hysteresis_waveform)
            ax_new.set_xlim(x_start, x_end)
            ax_new.set_ylim(0, -15)

            # drows the inset connector
            inset_connector(
                source_ax,
                target_ax,
                ax_new,
                [(x_start, 0), (x_end, 0)],
                [(x_start, 0), (x_end, 0)],
                color="k",
                linestyle="--",
                linewidth=0.5,
            )

            # adds a box on the figure
            add_box(
                target_ax,
                (x_start, 0, x_end, -15),
                edgecolor="k",
                linestyle="--",
                facecolor="none",
                linewidth=0.5,
                zorder=10,
            )
            ax_new.set_xlabel("Voltage Steps")
            ax_new.set_ylabel("Voltage (V)")

# Create a figure
fig = plt.figure(figsize=(14, 12))

# Define the GridSpec layout
gs = GridSpec(4, 4, figure=fig)

order = [['AFM'], [1], [0], [4, 'same', 6], [4, 'twin', 5], [2, 'inset', 3], ['hysteresis_loop']]

# List of axes indices in GridSpec for each subplot
subplot_specs = [(0, 2, 0, 2), # a (3D AFM)
                 (1, 2, 3, 4), # b (resonance frequency)
                 (1, 2, 2, 3), # c (waveform)
                 (0, 1, 2, 3), # d (real/imag)
                 (0, 1, 3, 4), # e (amp/phase)
                 (2, 4, 0, 2), # f (triangular waveform)
                 (2, 4, 2, 4)] # g (hysteresis)

# Create and set up subplots
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'AFM':
            image_path = "/home/jca92/Rapid-Fitting-BEPFM-NN/notebooks/assets/AFM_field.png"
            img = mpimg.imread(image_path)
            ax.imshow(img)
            ax.axis('off')  # Turn off axis for image subplot
        elif idx[0] == 'hysteresis_loop':
            raw_hysteresis_loop, voltage = dataset.get_hysteresis(
                                loop_interpolated=True, plotting_values=True)
            row = random.randint(0, 59)
            col = random.randint(0, 59)
            cycle = random.randint(0, 3)
            ax.plot(voltage.squeeze()*-1,
                       raw_hysteresis_loop[row, col, cycle, :].squeeze())
            ax.set_xlabel("Amplitude (Arb. U.)")
            ax.set_ylabel("Voltage (V)")
        elif len(idx) == 1:
            copy_axes_properties(axes[idx[0]], ax)
        else:
            copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
            

# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()

**Figure 1**: Schematic diagram of band-excitation piezoresponse force microscopy switching spectroscopy (BE-SS) **a** Artistic render of an AFM tip applying an electric field to the surface. **b** Band
of excited frequencies excited. The dashed line shows the cantilever resonance frequency. **c** Bandexcitation waveform used to excite the cantilever in time domain **d** Fast Fourier transform of a
single-cantilever resonance during band-excitation piezoresponse force microscopy – shown as real
and imaginary components. **e** Magnitude spectrum showing the amplitude and the phase of cantilever
resonance. **f** Bipolar-triangular waveform used to switch the ferroelectric. The inset shows where the
band-excitation waveform was applied in both the voltage-on and voltage-off states. **g** Example of a
typical piezoelectric hysteresis loop obtained during BE-SS.

## Figure 3

In [ ]:
LSQF_ = {'resampled': True,
                'raw_format': 'complex',
                'fitter': 'LSQF',
                'scaled': False,
                'output_shape': 'index',
                'measurement_state': 'all',
                'resampled_bins': 165,
                'LSQF_phase_shift': 1.5707963267948966,
                'NN_phase_shift': None,
                'noise': 0}

LSQF_Params = dataset.SHO_fit_results(state = LSQF_)

In [ ]:
set_seeds(seed=42)

device = 'cuda:1'
postprocessor = ComplexPostProcessor(dataset,device=device)


model_ = Multiscale1DFitter(SHO_fit_func_nn, # function 
                            dataset.frequency_bin, # x data
                            2, # input channels
                            4, # output channels
                            dataset.SHO_scaler, 
                            postprocessor,
                            device = device)




# instantiate the model
model = Model(model_, dataset, training=False, model_basename="SHO_Fitter_original_data",device = device)

model.load(
    "./Trained Models/SHO Fitter/2024-09-23_14-36-21_nn_benchmarks_noise/SHO_Fitter_model_optimizer_Adam_epoch_4_train_loss_0.040321211942850994.pth",
    device = device
)

#model.load_state_dict(torch.load("./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_599_train_loss_0.005903387442231178.pth"))

X_data, Y_data = dataset.NN_data()

X_data = X_data.to(device)
Y_data = Y_data.to(device)


# you can view the test and training dataset by replacing X_data with X_test or X_train
pred_data, scaled_param, parm = model.predict(X_data)

fig = BE_viz.SHO_switching_maps_test([LSQF_Params,parm], filename="Figure_15_NN_Switching_Maps",labels = ["LSQF","NN"])

In [ ]:
parm[:,0]

In [ ]:
1382400/64

In [ ]:
X_data.device

In [ ]:
X_data.shape

1. generate the above for the LSQF result
1. Put them into the combined figure to put the switching maps into figure 3
1. Copy the code to move the voltage curve into the placeholder

In [ ]:
fig

In [ ]:
X_data.shape

In [ ]:
parm.shape

In [ ]:
691200/(192*4)

In [ ]:
2764800/3600/4

In [ ]:
3600*192*2

In [ ]:
3600*384*4

In [ ]:
3600*192*4

In [ ]:
type(2.)

In [ ]:
axes = fig.axes

In [ ]:
true_state = {
    "fitter": "LSQF",
    "raw_format": "complex",
    "resampled": True,
    "scaled": True,
    "output_shape": "index",
    "measurement_state": "all",
}


fig = BE_viz.violin_plot_comparison_SHO(true_state, model, X_data, filename="Figure_16_Violin") 

# **FIGURE OUT HOW TO JUST COPY THE VIOLIN PLOT HERE, OR AT LEAST CHANGE THE PHASE**

In [ ]:
LSQF_ = {'resampled': True,
                'raw_format': 'complex',
                'fitter': 'LSQF',
                'scaled': True,
                'output_shape': 'index',
                'measurement_state': 'all',
                'resampled_bins': 165,
                'LSQF_phase_shift': 0, #1.5707963267948966,
                'NN_phase_shift': None,
                'noise': 0}


fig = BE_viz.violin_plot_comparison_SHO(LSQF_, model, X_data, filename="Figure_16_Violin_test") 

In [ ]:
!pwd

In [ ]:
axes.extend(fig.axes)

In [ ]:
dataset.noise

In [ ]:
# sets the phase shift of the dataset
dataset.NN_phase_shift = np.pi/2
dataset.LSQF_phase_shift = np.pi/2
dataset.measurement_state = "all"

# sets the true state which to compare the results.
true_state = {
    "fitter": "LSQF",
    "raw_format": "complex",
    "resampled": True,
    "scaled": True,
    "output_shape": "index",
    "measurement_state": "all",
}

# sets the state of the output data
out_state = {"scaled": True, "raw_format": "magnitude spectrum"}

#out_state = {"scaled": True, "raw_format": "complex"}


# sets the number of examples to get
n = 1

LSQF = BE_viz.get_best_median_worst(
    true_state,
    prediction={"fitter": "LSQF"},
    out_state=out_state,
    SHO_results=True,
    n=n,
)

NN = BE_viz.get_best_median_worst(
    true_state, prediction=model, out_state=out_state, SHO_results=True, n=n
)

data = (LSQF, NN)
names = ["LSQF", "NN"]

fig = BE_viz.SHO_Fit_comparison(
    data,
    names,
    model_comparison=[model, {"fitter": "LSQF"}],
    out_state=out_state,
    filename="Figure_14_LSQF_NN_bmw_comparison_test",
    # display_results = None
)

In [ ]:
# axes.extend(fig.axes)

In [ ]:
axes=fig.axes

In [ ]:
axes

In [ ]:
axes[1].figure

In [ ]:
for line in axes[1].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        print(min(line.get_xdata()/1e6))
        plt.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
plt.show()

In [ ]:
axes

In [ ]:
for i in range(12):
    print(axes[i].get_xlim())

In [ ]:
len(axes)

In [ ]:
from m3util.viz.text import number_to_letters

number_to_letters(0+8)

In [ ]:
axes

In [ ]:
# (a is fine)          (b should be d)
# (c should be b)      (d should be e)
# (e should be c)      (f is fine)


# so it should be 

# a   d
# b   e
# c   f

In [ ]:
axes[3].get_ylim()

In [ ]:
for col in range(2):
    for row in range(3): 
        print(row,col, 2*row+col)

In [ ]:
for line in axes[0].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        plt.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
        


In [ ]:
ScalarFormatter(8.0).

In [ ]:
format(8e-3,".3f")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

# Create some data
x = np.linspace(0, 10, 100)
y = np.sin(x) * 1e-3  # Data on the order of 1e-3

# Create the figure and subplot
fig, ax = plt.subplots()

# Plot the data
ax.plot(x, y)

# Custom formatter for the y-axis to scale by 1e-3
def y_formatter(y, pos):
    return f"{y * 1e3:.1f}"  # Multiply by 1e3 to show scaled values

# Apply the custom formatter to the y-axis
ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))

# Add an indicator for scaling on the y-axis (e.g., x10^-3)
ax.set_ylabel(r"y ($\times 10^{-3}$)")

# Add labels and title
ax.set_xlabel("x")
ax.set_title("Custom Tick Formatting with Scaled y-axis")

# Automatically adjust tick positions and labels
plt.tight_layout()



labelfigs(ax,
    number=0,
    loc ='tl',
    size=15,
    inset_fraction=(0.08,0.2),
    style = 'b'
    )


In [ ]:
print(axes[0].get_legend_handles_labels())
print(axes[6].get_legend_handles_labels())
# ax_.legend(lines + lines2, labels + labels2, loc="upper right")


In [ ]:
import pandas as pd
import seaborn as sns
import itertools
from torch import nn


from m3util.viz.layout import imagemap, FigDimConverter, subfigures
from m3util.viz.text import number_to_letters

from matplotlib.ticker import ScalarFormatter
#from matplotlib.ticker import FormatStrFormatter
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import EngFormatter
#from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable



def y_formatter(y, pos):
    return f"{y * 1e3:.1f}"  # Multiply by 1e3 to show scaled values

def copy_axes_properties(row,col,source_ax, target_ax, secondary_ax, ax_lims):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    #target_ax.set_xlabel(source_ax.get_xlabel())
    # uncomment out below 
    target_ax.set_xlim([1.2,1.4])

    if row == 2: # if i in [4,5]:
        target_ax.set_xlabel('Frequency (MHz)',fontsize=20)
    else:
        target_ax.set_xlabel("")
        #target_ax.set_xticks([])
        target_ax.xaxis.set_ticklabels([])

   #target_ax.set_ylim(ax_lims.get_ylim() if max(ax_lims.get_ylim()) > max(source_ax.get_ylim()) else source_ax.get_ylim())

    if col == 0:    
        #target_ax.set_ylim(source_ax.get_ylim())
        target_ax.set_ylabel(source_ax.get_ylabel(),fontsize=20)
    else:
        #target_ax.set_ylim(ax_lims.get_ylim() if max(ax_lims.get_ylim()) > max(source_ax.get_ylim()) else source_ax.get_ylim())
        target_ax.set_ylabel("")
        target_ax.yaxis.set_ticklabels([])

        
    if row == 0:
        target_ax.set_ylim([-0.5e-3,8.0e-3])
    elif row == 1: 
        target_ax.set_ylim([-0.1e-2,2.1e-2])
    elif row == 2: 
        target_ax.set_ylim([-0.1e-2,2.1e-2])    
    
    #target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
        
    # Handle twin axes if present
    #if secondary_ax:
    ax_twin = target_ax.twinx()
    ax_twin.set_ylim(secondary_ax.get_ylim())
    #ax_twin.set_yticks([-np.pi,0,np.pi],labels =["$-\pi$","0","$\pi$"])

    if col == 0: #if i % 2 == 0: 
        ax_twin.set_ylabel("")
        ax_twin.yaxis.set_ticklabels([])

        # ax_twin.set_yticks([])
        
        set_sci_notation_label(
                target_ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                textsize = 10, offset_points = (0,30)
            )
    else:
        #target_ax.set_ylabel("")
        ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize = 20)
        ax_twin.set_yticks([-3,-2,-1,0,1,2,3])

    
    #ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize = 25)


    for line in secondary_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties for the twin axis
        ax_twin.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                        linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # # Copy legends
    # if source_ax.get_legend():
    #     target_ax.legend(fontsize='large')

    # if secondary_ax and secondary_ax.get_legend():
    #     ax_twin.legend(fontsize='large')
        
    
    
    #target_ax.set_xlim(min(line.get_xdata())/1e6,max(line.get_xdata())/1e6)
    target_ax.tick_params(axis='x',labelsize=20)
    target_ax.tick_params(axis='y',labelsize=20)
    #target_ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    if row == 0 and col == 0: 
        target_ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))



    #ax_twin.tick_params(axis='x',labelsize=15,length = 10, width = 2)
    ax_twin.tick_params(axis='x',labelsize=20)

    ax_twin.tick_params(axis='y',labelsize=20)
    
    plt.tight_layout()

# Create a figure

fig = plt.figure(figsize=(24, 24))


# Define the GridSpec layout
gs = GridSpec(60, 40, figure=fig)

#gs = GridSpec(6, 4, figure=fig)


# order = [[42, 'twin', 48],
#          [43, 'twin', 49],
#          [44, 'twin', 50],
#          [45, 'twin', 51],
#          [46, 'twin', 52],
#          [47, 'twin', 53],
#          ['violin'],
#         ]

# order = [[0, 'twin', 6],
#          [1, 'twin', 7],
#          [2, 'twin', 8],
#          [3, 'twin', 9],
#          [4, 'twin', 10],
#          [5, 'twin', 11],
#          ['violin'],
#          ['voltage_curve'],
#          ['switching_maps']
#         ]


order = [['SHO_fit_comp'],
         ['violin'],
         ['voltage_curve'],
         ['switching_maps']
        ]


# # List of axes indices in GridSpec for each subplot
# subplot_specs = [(0, 8, 0, 8), # a
#                  (0, 8, 10, 18), # b
#                  (10, 18, 0, 8), # c
#                  (10, 18, 10, 18), # d
#                  (20, 28, 0, 8), # e
#                  (20, 28, 10, 18), # f
#                  (0, 20, 20, 40), # g
#                  (25, 35, 20, 40), # h
#                  (35, 80, 0, 60), #bottom 
                 
#                 ]


# # List of axes indices in GridSpec for each subplot
# subplot_specs = [(0, 1, 0, 1), # a
#                  (0, 1, 1, 2), # b
#                  (1, 2, 0, 1), # c
#                  (1, 2, 1, 2), # d
#                  (2, 3, 0, 1), # e
#                  (2, 3, 1, 2), # f
#                  (0, 2, 2, 4), # g
#                  (2, 3, 2, 4), # h
#                  (3, 8, 0, 6), #bottom 
#                 ]

# # List of axes indices in GridSpec for each subplot
# subplot_specs = [(0, 3, 0, 2 ), # top left: SHO fit comparisons 
#                  (0, 2, 2, 4), # g
#                  (2, 3, 2, 4), # h
#                  (3, 8, 0, 6), #bottom 
#                 ]

subplot_specs = [(0, 30, 0, 20 ), # top left: SHO fit comparisons 
                 (0, 15, 20, 40), # g
                 (17, 27, 20, 40), # h
                 (30, 80, 0, 60), #bottom 
                ]


renderViolin = True
renderVoltage = True
renderSwitchingMaps = True
renderBMWComp = True
# Create and set up subplots
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'violin':
            if renderViolin: 
                df = pd.DataFrame()

                # scales the parameters
                scaled_param = dataset.SHO_scaler.transform(parm)

                # gets the parameters from the SHO LSQF fit
                true = dataset.SHO_fit_results(phase_shift=0).reshape(-1, 4)

                # Builds the dataframe for the violin plot
                true_df = pd.DataFrame(
                    true, columns=["Amplitude", "Resonance", "Q-Factor", "Phase"]
                )
                predicted_df = pd.DataFrame(
                    scaled_param, columns=["Amplitude",
                                        "Resonance", "Q-Factor", "Phase"]
                )

                # merges the two dataframes
                df = pd.concat((true_df, predicted_df))

                # adds the labels to the dataframe
                names = [true, scaled_param]
                names_str = ["LSQF", "NN"]
                labels = ["A", "\u03C9", "Q", "\u03C6"]

                # adds the labels to the dataframe
                for j, name in enumerate(names):
                    for i, label in enumerate(labels):
                        dict_ = {
                            "value": name[:, i],
                            "parameter": np.repeat(label, name.shape[0]),
                            "dataset": np.repeat(names_str[j], name.shape[0]),
                        }

                        df = pd.concat((df, pd.DataFrame(dict_)))
                
                # Reset index to handle potential duplicated columns or indices
                df = df.reset_index(drop=False)
                
                # plots the data
                sns.violinplot(
                    data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax
                )

                # labels the figure and does some styling
                #labelfigs(ax, 0, style="b")
                labelfigs(ax, string_add = 'g', loc ='tl',size=22, style="b", inset_fraction=(0.05,0.15))

                ax.set_ylabel("Scaled SHO Results",fontsize=20)
                ax.set_xlabel("")
                
                ax.tick_params(axis='x',labelsize=20)
                ax.tick_params(axis='y',labelsize=20)
                #ax.set_ylim([-10,6])
                ax.set_yticks(np.linspace(-6,6,7))


                # Get the legend associated with the plot
                legend = ax.get_legend()
                legend.set_title("")
                plt.setp(legend.get_texts(), fontsize=20) # Set the label size
                
        elif idx[0] == "voltage_curve":
            if renderVoltage:
                #pass
                _,voltage = dataset.get_hysteresis()
                voltage = dataset.roll_hysteresis(voltage)

                # # Get indices of the voltage steps to plot
                inds = np.linspace(0, len(voltage) - 1, 9, dtype=int)

                # plots the voltage
                ax.plot(voltage, "k")
                ax.set_ylabel("Voltage (V)",fontsize=20)
                ax.set_xlabel("Step",fontsize=20)
                ax.set_xticks(np.linspace(0,100,11))
                ax.set_xlim([-4,100])
                ax.set_ylim([-20,20])
                ax.set_yticks([-15,0,15])
                ax.tick_params(axis='x',labelsize=20)
                ax.tick_params(axis='y',labelsize=20)

                label_marker_symbols_for_plt = ["o", "v", "^", ">", "<", "s","P", "D","*"]

                
                # Plot the data with different markers
                for i, ind in enumerate(inds):
                    # this adds the labels to the graphs
                
                    ax.plot(ind, voltage[ind], label_marker_symbols_for_plt[i], 
                            color="k", markersize=20)
                    vshift = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.25

                    # positions the location of the labels
                    if voltage[ind] - vshift - 0.15 < ax.get_ylim()[0]:
                        vshift = -vshift / 2

                    # # adds the text to the graphs
                    ax.text(ind, voltage[ind] - vshift,
                            number_to_letters(i + 8), color="k", fontsize=25)
                    
                    # labelfigs(ax, number = i+8, loc = 'tl', size=20,style='b',
                    #           inset_fraction=(ind,voltage[ind]-vshift))
                plt.tight_layout()
                
                labelfigs(ax,
                    string_add='h',
                    loc ='tl',
                    size=15, #22
                    inset_fraction=(0.12,0.04),
                    style = 'b'
                    )

        elif idx[0] == "switching_maps": # see BE_viz.SHO_switching_maps_test for reference
            if renderSwitchingMaps:
            # pass
                

                # # # Get indices of the voltage steps to plot
                # inds = np.linspace(0, len(voltage) - 1, 9, dtype=int)
            
                clims=[
                    (0, 1.4e-4),  # amplitude
                    (1.31e6, 1.33e6),  # resonance frequency
                    (-230, -160),  # quality factor
                    (-np.pi, np.pi), # phase
                ]  
            
                number_of_steps = 9
                SHO_ = [LSQF_Params,parm]
                comp_number = len(SHO_)

                rows = np.ceil(number_of_steps * comp_number / 3)
                cols = 3
                
                
                fig_width = 6.5
                intra_gap=0.02  # gap between the graphs,
                inter_gap=0.05  # gap between the graphs,
                cbar_gap=0.4  # gap between the graphs of colorbars
                cbar_space=1.3  # space on the right where the cbar is not
                voltage_plot_height=1.25  # height of the voltage plot
                colorbars = True

                        
                # calculates the size of the embedding image
                embedding_image_size = (
                    fig_width
                    - (inter_gap * (cols - 1))
                    - intra_gap * 3 * cols
                    - cbar_space * colorbars
                ) / (cols * 4)
                
                
                # calculates the figure height based on the image details
                fig_height = (
                    rows * (embedding_image_size + inter_gap)
                    + voltage_plot_height
                    + 0.33
                    + inter_gap * (comp_number - 1)
                )

                # defines a scalar to convert inches to relative coordinates
                fig_scalar = FigDimConverter((fig_width, fig_height))
                fig = plt.figure(figsize=(fig_width, fig_height))

                ax2 = [] 
                
                    # Define the position and size of the voltage plot
                pos_inch = [
                    0.33,
                    fig_height - voltage_plot_height,
                    fig_width - 0.33,
                    voltage_plot_height,
                ]
                
                # adds the plot for the voltage
                #ax2.append(fig.add_axes(fig_scalar.to_relative(pos_inch)))
                
                #len(ax) here should be 1
                
                
                
                # resets the x0 position for the embedding plots
                pos_inch[0] = 0
                pos_inch[1] -= embedding_image_size + 0.33

    
                # sets the embedding size of the image
                pos_inch[2] = embedding_image_size
                pos_inch[3] = embedding_image_size
                
                    # This makes the figures
                for k, _SHO in enumerate(SHO_):
                    # adds the embedding plots
                    for i in range(number_of_steps):
                        # loops around the amp, phase, and freq
                        for j in range(4):
                            # adds the plot to the figure
                            ax2.append(fig.add_axes(fig_scalar.to_relative(pos_inch)))

                            # adds the inter plot gap
                            pos_inch[0] += embedding_image_size + intra_gap

                        # if the last column in row, moves the position to the next row
                        if (i + 1) % cols == 0 and i != 0:
                            # resets the x0 position for the embedding plots
                            pos_inch[0] = 0

                            # moves the y0 position to the next row
                            pos_inch[1] -= embedding_image_size + inter_gap

                            if (i + 1) % (cols * comp_number) == 0 and comp_number > 1:
                                pos_inch[1] -= inter_gap

                        else:
                            # adds the small gap between the plots
                            pos_inch[0] += inter_gap
                # _,voltage = dataset.get_hysteresis()
                # voltage = dataset.roll_hysteresis(voltage)
                # gets the index of the voltage steps to plot
                #inds = np.linspace(0, len(voltage) - 1, number_of_steps, dtype=int)
                
                
                for k, _SHO in enumerate(SHO_):
                    # converts the data to a numpy array
                    if isinstance(_SHO, torch.Tensor):
                        _SHO = _SHO.detach().numpy()

                    print(_SHO.shape)
                    _SHO = _SHO.reshape(dataset.num_pix,
                                        dataset.voltage_steps, 4)

                    # get the selected measurement cycle
                    _SHO = dataset.get_measurement_cycle(_SHO, axis=1)

                    names = ["A", "\u03C9", "Q", "\u03C6"]

                    for i, ind in enumerate(inds):
                        axis_start = int(
                            (i % cols) * 4
                            + ((i) // cols) * (comp_number * cols * 4)
                            + k * (cols * 4)
                            + 1
                        )

                        # loops around the amp, resonant frequency, and Q, Phase
                        for j in range(4):
                            imagemap(
                                ax2[axis_start + j-1],
                                _SHO[:, ind, j],
                                colorbars=False,
                                cmap="viridis",
                            )

                            if i // rows == 0 and k == 0:
                                labelfigs(
                                    ax2[axis_start + j-1],
                                    string_add=names[j],
                                    loc="cb",
                                    size=5,
                                    inset_fraction=(0.2, 0.2),
                                )

                            ax2[axis_start + j-1].images[0].set_clim(clims[j])

                            if k == 0:
                                labelfigs(
                                    ax2[axis_start + j-1],
                                    string_add=str(i + 1),
                                    size=5,
                                    loc="bl",
                                    inset_fraction=(0.2, 0.2),
                                )

                            if (axis_start + j) % (4 * cols) == 1:
                                ax2[axis_start + j-1].set_ylabel(names_str[k])
                # if add colorbars
                # if colorbars:
                #     # builds a list to store the colorbar axis objects
                #     bar_ax = []

                    
                #     # defines a scalar to convert inches to relative coordinates
                #     fig_scalar = FigDimConverter((fig_width, fig_height))
                            
                    
                #     # gets the voltage axis position in ([xmin, ymin, xmax, ymax]])
                #     voltage_ax_pos = fig_scalar.to_inches(
                #         np.array(ax2[0].get_position()).flatten()
                #     )
                    
                #     fmt = ScalarFormatter(useMathText=True)
                #     fmt.set_powerlimits((0, 0))
                #     # loops around the 4 axis
                #     for i in range(4):
                #         # calculates the height and width of the colorbars
                #         cbar_h = (voltage_ax_pos[1] -
                #                 inter_gap - 2 * intra_gap - 0.33) / 2
                #         cbar_w = (cbar_space - inter_gap - 2 * cbar_gap) / 2

                #         # sets the position of the axis in inches
                #         pos_inch = [
                #             voltage_ax_pos[2] - (2 - i % 2) *
                #             (cbar_gap + cbar_w) + inter_gap,
                #             voltage_ax_pos[1] - (i // 2) *
                #             (inter_gap + cbar_h) - 0.33 - cbar_h,
                #             cbar_w - 0.02,
                #             cbar_h - 0.1,
                #         ]

                #         # adds the plot to the figure
                #         bar_ax.append(fig.add_axes(fig_scalar.to_relative(pos_inch)))

                #         # adds the colorbars to the plots 
                #         fmt = ScalarFormatter(useMathText=True)
                #         fmt.set_powerlimits((0, 0))
                #         cbar = plt.colorbar(ax2[i + 1].images[0],
                #                             cax=bar_ax[i], format=fmt)
                #         cbar.set_label(names[i])  # Add a label to the colorbar

            
                    # for line in ax2.get_lines():
                    #     label = line.get_label() if line.get_label() != '_nolegend_' else None
                    #     # Copying line properties like color, linestyle, marker, etc.
                    #     ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                    #                 linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
                #ax.inset_axes([0,0,60,60])
                #ax.imshow(ax2[0][])
                #ax.add_image(ax2[0].findobj()[0])
                #ax.imshow(ax2[0].get_images()[0].get_array().data)
                #fig3 = plt.figure(figsize = (fig_width, fig_height))
                
                label_marker_symbols = ["\u25CF", "\u25BC", "\u25B2", "\u25BA", "\u25C0", "\u25A0","\u271A", "\u25C6","\u2605"]
                label_marker_symbols_counter = 0
                
                
                labels = ['i','j','k','l','m','n','o','p','q']
                label_counter = 0
                names = ['Amplitude', "Resonance","Quality Factor","Phase"]
                clims=[
                        (0, 1.4e-4),  # amplitude
                        (1.31e6, 1.33e6),  # resonance frequency
                        (-240, -160),  # quality factor
                        (-np.pi, np.pi),  # phase
                    ],  # phase limits
                #fmt = EngFormatter(useMathText=True, places = 2)
                fmt = ScalarFormatter(useMathText=True)
                fmt.set_powerlimits((0, 0))
                #fmt.format("%1.2f")
                # defines a scalar to convert inches to relative coordinates
                fig_scalar = FigDimConverter((1/6, 1/6))
                #fig_scalar = FigDimConverter((fig_width, fig_height))
                
                for row in range(6):
                    for col in range(12):
                        inset_ax = ax.inset_axes([-0.06+(col/11.8)+np.floor(col/4)/256,1-(row+1)/6.1-row/192-np.floor(row/2)/96,1/6.1,1/6.1])

                        inset_ax.imshow(ax2[(12*row)+col].get_images()[0].get_array().data,clim = clims[0][int(np.floor(col % 4))])
                        if col == 0:
                            if row % 2 == 0: 
                                inset_ax.text(30,10,ax2[0].get_ylabel(),color = "white",size=15,ha = "center", va = "center")
                            
                            else:
                                inset_ax.text(30,10,ax2[12].get_ylabel(),color = "white", size=15,ha = "center", va = "center")
                        
                        if row % 2 == 0 and col % 4 == 0: 
                                inset_ax.text(10,10, label_marker_symbols[label_marker_symbols_counter], color = "white", size = 24, ha = "center", va = "center")
                                label_marker_symbols_counter+=1
                        elif row % 2 == 1 and (col + 1) % 4 == 0:
                            inset_ax.text(50,50, labels[label_counter], color = "white", weight = 'bold', size = 15,ha = "center", va = "center")
                            label_counter+=1 
                            
                        if row == 5:
                            # if int(np.floor(col % 4)) != 3: 
                            # fmt = EngFormatter(useMathText=True, places = 2)
                            # #fmt.set_powerlimits((0, 0))
                            
                            fmt = ScalarFormatter(useMathText=True)
                            fmt.set_powerlimits((0, 0))
                                        
                            bar_ax = []
                            pos_inch = [-3.2e-3 + (col/70.5)+np.floor(col/4)/1800, -0.008, 1/73.5, 1/500  ] #fills axes
                           # pos_inch = [-3e-3 + (col/70), -0.008, 1/85, 1/500  ] #1/120 third term 
                            #pos_inch = [-3e-3 + (col/70) + np.floor(col/4)/1800, -0.008, 1/85, 1/500  ] 

                            #pos_inch = [-0.06 + col/11.8 + np.floor(col/4/256),-0.008, 1/6.1,1/100]
                            bar_ax.append(ax.inset_axes(fig_scalar.to_relative(pos_inch)))
                            #bar_ax.append(ax.inset_axes((pos_inch)))
                            #bar_ax.append(inset_axes(ax, width =  ))
                            
                            # if int(np.floor(col % 4)) ==0 != 3: 

                            #     cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], 
                            #                         format = fmt, 
                            #                         ticks = np.linspace(np.min(clims[0][int(np.floor(col % 4))]),
                            #                                             np.max(clims[0][int(np.floor(col % 4))]),2) #5
                            #                         )
                                # exponent_axis = np.floor(np.log10(abs(clims[0][int(np.floor(col % 4))][1]))).astype(int)
                            
                                
                                                            
                                # cbar.ax.text(0.55,0.5, f'$\\times 10^{{{exponent_axis}}}$',size=10)
                                #cbar.ax.xaxis.offsetText.set_visible(False)
                            def fmt(x, pos):
                                    a, b = '{:.1e}'.format(x).split('e')
                                    b = int(b)
                                    if abs(b) >2: 
                                        return r'${} \times 10^{{{}}}$'.format(a, b)
                                    else: 
                                        return float(a)*10**b
                                    
                                    
                                    
                            if int(np.floor(col % 4)) == 0:
                                
                                
                                                                
                                cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], 
                                                    format = FuncFormatter(fmt), 
                                                    ticks = np.linspace(np.min(clims[0][int(np.floor(col % 4))]),
                                                                        np.max(clims[0][int(np.floor(col % 4))]),2), #5
                                                  
                                                    )
                                #cbar.set_ticks(np.min(clims[0][int(np.floor(col % 4))]),np.min(clims[0][int(np.floor(col % 4))]))
                                cbar.ax.get_xticklabels()[0].set_horizontalalignment('left')
                                cbar.ax.get_xticklabels()[1].set_horizontalalignment('right')

                            elif int(np.floor(col % 4)) == 1:
                                
                                def fmt_Resonance(x, pos): #need more digits
                                    a, b = '{:.2e}'.format(x).split('e')
                                    b = int(b)
                                    return r'${} \times 10^{{{}}}$'.format(a, b)
                                    
                                
                                cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], 
                                                    format = FuncFormatter(fmt), #fmt, 
                                                    ticks = np.linspace(np.min(clims[0][int(np.floor(col % 4))]),
                                                                        np.max(clims[0][int(np.floor(col % 4))]),2) #5
                                                    )
                            elif int(np.floor(col % 4)) == 2:
                                cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], 
                                                    format = FuncFormatter(fmt), 
                                                    ticks = np.linspace(np.min(clims[0][int(np.floor(col % 4))]),
                                                                        np.max(clims[0][int(np.floor(col % 4))]),2) # 5
                                                    )
                            else:
                                cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], 
                                                    format = FuncFormatter(fmt), 
                                                    ticks = np.linspace(-3,3,2)
                                                    ) 
                            
                            cbar.ax.get_xticklabels()[0].set_horizontalalignment('left')
                            cbar.ax.get_xticklabels()[1].set_horizontalalignment('right')
                            cbar.ax.tick_params(labelsize = 10)
                            cbar.set_label(names[int(np.floor(col % 4))],size=15)  # Add a label to the colorbar


                            
                            # if int(np.floor(col % 4)) != 3: 
                            # #     #divider = make_axes_locatable(inset_ax)

                            # #    # cax = divider.append_axes("bottom", size="5%", pad=0.0)

                                
                            # #     #cbar = plt.colorbar(inset_ax.images[0],orientation = 'horizontal', cax = cax, 
                            # #     # cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], 
                            # #     #                     format = fmt, pad = 0.2,
                            # #     #                     ticks = np.linspace(np.min(clims[0][int(np.floor(col % 4))]),
                            # #     #                                         np.max(clims[0][int(np.floor(col % 4))]),2) #5
                            # #     #                     )
                            # #    # cbar.update_ticks()

                            #     exponent_axis = np.floor(np.log10(abs(clims[0][int(np.floor(col % 4))][1]))).astype(int)
                            
                                
                                                            
                            #     cbar.ax.text(1.005,0.5, f'$\\times 10^{{{exponent_axis}}}$',size=10, ha = "left", va = "center",
                            #                  transform = cbar.ax.transAxes)
                            #     cbar.ax.xaxis.offsetText.set_visible(False)
                                
                            # cbar.set_label(names[int(np.floor(col % 4))],size=15)  # Add a label to the colorbar
                            # cbar.ax.tick_params(labelsize = 10)
                            #cbar.ax.xaxis.offsetText.set_visible(False)
                            
                           # fig.subplots_adjust(bottom=0.2)  # Add space at the bottom for the colorbar


                            
                            # Image size of 994508x1964 pixels or 525540 if cbar.ax.text(0.55,0.5) 517188 515912
                            # Image size of 994486x1964 pixels
                            #         16x16: 663012x1322 pixels
                            
                        # ax.inset_axes.cbar(clims[0][int(np.floor(col % 4))])
                            
                            #cbar.ax.tick_params(labelsize=15)

                            # inset_ax.text(0.7,-0.2, names[int(np.floor(col % 4))], color = 'black',size = 15, 
                            #             horizontalalignment='center',
                            #             verticalalignment='center',
                            #             transform=inset_ax.transAxes,
                            #             )

                            
                            
                            
                                                    
    
                        


                        inset_ax.axis("off")
                
                ax.axis("off")       
                
                # inset_ax = ax.inset_axes([-0.02, 1-1/12, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[0].get_images()[0].get_array().data)
                # inset_ax.axis("off")
                
                # inset_ax = ax.inset_axes([-0.02+(1/23), 1-1/12, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[1].get_images()[0].get_array().data)
                # inset_ax.axis("off")
                
                # inset_ax = ax.inset_axes([-0.02+(2/23), 1-1/12, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[2].get_images()[0].get_array().data)
                # inset_ax.axis("off")

                # inset_ax = ax.inset_axes([-0.02, 1-2/12-1/192, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[12].get_images()[0].get_array().data)
                # inset_ax.axis("off")
                
                # inset_ax = ax.inset_axes([-0.02, 1-3/12-2/192, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[24].get_images()[0].get_array().data)
                # inset_ax.axis("off")
                
                # inset_ax = ax.inset_axes([-0.02, 1-4/12-3/192, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[36].get_images()[0].get_array().data)
                # inset_ax.axis("off")
                
                # inset_ax = ax.inset_axes([-0.02, 1-5/12-4/192, 1/12, 1/12])  # [x, y, width, height] in relative coordinates
                # inset_ax.imshow(ax2[48].get_images()[0].get_array().data)
                # inset_ax.axis("off")
                
                
                #for i in range(len(ax2)):
                #   ax = fig.add_subplot()
                    
                        
                #      ax.imshow(ax2[i].get_images()[0].get_array().data)
                #      ax.text(30,5,i,color="white", size = "large")
                #      ax.axis("off")
                    #ax.add_child_axes(ax2[i])
                #    ax.ins
                # Render the figure as an image
                # fig.tight_layout()
                # canvas = FigureCanvas(ax2[0].figure)
                # canvas.draw()  # Render the figure onto the canvas
                # image = np.frombuffer(canvas.tostring_rgb(), dtype='uint8')
                # image = image.reshape(canvas.get_width_height()[::-1] + (3,))
                
                # ax.imshow(image)
                # ax.axis("off")
            
        else:
            if renderBMWComp:
                ax.axis("off")       

                # Defines the color palettes for the plots
                color_palette = {
                    "LSQF_A": "#003f5c",  # dark blue
                    "LSQF_P": "#444e86",  # bluish purple
                    "NN_A": "#955196",  # purple
                    "NN_P": "#dd5182",  # pinkish red
                    "real": "#ff6e54",  # orange
                    "imag": "#ffa600",  # yellow-orange
                    "mag": "#2f9eaa",  # cyan
                    "phase": "#66c21f",  # green
                }

                
                
                # sets the phase shift of the dataset
                dataset.NN_phase_shift = np.pi/2
                dataset.LSQF_phase_shift = np.pi/2
                dataset.measurement_state = "all"

                # sets the true state which to compare the results.
                true_state = {
                    "fitter": "LSQF",
                    "raw_format": "complex",
                    "resampled": True,
                    "scaled": True,
                    "output_shape": "index",
                    "measurement_state": "all",
                }

                # sets the state of the output data
                out_state = {"scaled": True, "raw_format": "magnitude spectrum"}

                #out_state = {"scaled": True, "raw_format": "complex"}


                # sets the number of examples to get
                n = 1

                LSQF = BE_viz.get_best_median_worst(
                    true_state,
                    prediction={"fitter": "LSQF"},
                    out_state=out_state,
                    SHO_results=True,
                    n=n,
                )

                NN = BE_viz.get_best_median_worst(
                    true_state, prediction=model, out_state=out_state, SHO_results=True, n=n
                )
                # from BE_viz.SHO_fit_comparison 
                
                data = (LSQF, NN)
                names = ["LSQF", "NN"]
                model_comparison=[model, {"fitter": "LSQF"}]
                out_state=out_state
                gaps=(0.8, 0.1)
                size=(1.25, 1.25)
                #display_results="all"
                
                # Get the number of fits from the length of the data list
                num_fits = len(data)
                
                # Create subplots for the comparison
                fig_SHO_comp, ax_SHO_comp = subfigures(3, num_fits, gaps=gaps, size=size)
                
                list_ax_ = []
                list_ax1 = [] 
                            
                # Loop through each fit and the associated data
                for step, (data, name) in enumerate(zip(data, names)):
                    # Unpack the data (true, predicted values, indices, etc.)
                    d1, d2, x1, x2, label,full_labels, index1, mse1, params = data

                    # Loop through datasets for comparison (true vs. predicted data)
                    for bmw, (true, prediction, error, SHO, index1) in enumerate(
                        zip(d1, d2, mse1, params, index1)
                    ):
                        # Initialize dictionaries for errors and SHO parameters
                        errors = {}
                        SHOs = {}

                        # Determine the subplot index
                        index = bmw * num_fits + step
                        ax_ = ax_SHO_comp[index]

                        # Plot predicted amplitude and phase
                        ax_.plot(
                            x2,
                            prediction[0].flatten(),
                            color=color_palette[f"{name}_A"],
                            label=f"{name} {label[0]}",
                        )
                        ax1 = ax_.twinx()
                        ax1.plot(
                            x2,
                            prediction[1].flatten(),
                            color=color_palette[f"{name}_P"],
                            label=f"{name} {label[1]}",
                        )

                        # Plot true amplitude and phase
                        ax_.plot(
                            x1,
                            true[0].flatten(),
                            "o",
                            color=color_palette["LSQF_A"],
                            label=f"Raw {label[0]}",
                        )
                        ax1.plot(
                            x1,
                            true[1].flatten(),
                            "o",
                            color=color_palette["LSQF_P"],
                            label=f"Raw {label[1]}",
                        )

                        # Store errors and SHO parameters for the current model
                        errors[name] = error
                        SHOs[name] = SHO
                        
                        
                        if model_comparison[step] is not None:
                            # Get SHO parameters from the comparison model
                            pred_data, params, labels = BE_viz.get_SHO_params(
                                index1, model=model_comparison[step], out_state=out_state
                            )

                            # Determine the color prefix based on model type (NN or LSQF)
                            if isinstance(model_comparison[step], nn.Module):
                                color = "NN"
                            elif isinstance(model_comparison[step], dict):
                                color = "LSQF"

                            # Store errors and SHO parameters for the comparison model
                            errors[color] = BE_viz.get_mse_index(
                                index1, model_comparison[step]
                            )
                            SHOs[color] = np.array(params).squeeze()

                            # Plot the comparison data
                            ax_.plot(
                                x2,
                                pred_data.squeeze()[0].flatten(),
                                color=color_palette[f"{color}_A"],
                                label=f"{color} {labels[0]}",
                            )
                            ax1.plot(
                                x2,
                                pred_data.squeeze()[1].flatten(),
                                color=color_palette[f"{color}_P"],
                                label=f"{color} {labels[1]}",
                            )
                            
                        # Set the x-axis label (Frequency in MHz)
                        ax_.set_xlabel("Frequency (MHz)")
                        
                        if (
                            "raw_format" in out_state.keys()
                            and out_state["raw_format"] == "magnitude spectrum"
                        ):
                            ax_.set_ylabel("Amplitude (Arb. U.)")
                            ax1.set_ylabel("Phase (rad)")
                        else:
                            ax_.set_ylabel("Real (Arb. U.)")
                            ax1.set_ylabel("Imag (Arb. U.)")
                            
                        
                        # Add legend for the last fit
                        if index < num_fits:
                            lines, labels = ax_.get_legend_handles_labels()
                            lines2, labels2 = ax1.get_legend_handles_labels()
                            ax_.legend(lines + lines2, labels + labels2, loc="upper right")
                        
                        set_sci_notation_label(ax_,axis="x",corner = "bottom right")
                        
                        
                        
                        list_ax_.append(ax_)
                        list_ax1.append(ax1)
                        
                        plt.tight_layout()
                        # ax.add_child_axes(ax_)
                        # ax.add_child_axes(ax1)
                        
                        # THE ISSUE IS THAT THERE ARE TWO AXES TO PLOT. 
                        # HOW DO I MOVE THEM BOTH TO THE GRIDSPEC AXIS? 
                        # IDEALLY, THE ABOVE CODE WOULD DETERMINE THE SPACING
                        # BUT IDK. 
                        #row,col       # axes numbers
                        # (0,0) (0,1)    (0,1)  (2,3) 
                        # (1,0) (1,1)    (4,5)  (6,7) 
                        # (2,0) (2,1)    (8,9)  (10,11)
                        
                        
                        # (row, col) iterations: 
                        # (0,0) => (top, left) => (2*row + col) = 0
                        # (0,1) => (top, right) => (2*row + col) = 1
                        # (1,0) => (middle, left) => (2*row + col) = 2
                        # (1,1) => (middle, right) => (2*row+col) = 3 
                        # (2,0) => (bottom, left)
                        # (2,1) => (bottom, right)
                        
                        
                        # it should be 

                        # a  d
                        # b  e
                        # c  f
                                            
                        # what it is now:  
                        # a  c        0   2
                        # e  b  ==>   4   1
                        # d  f        3   5
                        
                        
                        # so in index order it goes=> should be : 
                        # 0 => 0
                        # 1 => 3
                        # 2 => 1
                        # 3 => 5
                        # 4 => 2
                        # 5 => 5
                        
                        
                axes_index = [0,3,1,4,2,5]
                        
                for row in range(3):
                    for col in range(2):
                        inset_ax = ax.inset_axes([(col/2)-col*0.05,1-(row+1)/3.25,1/3.25,1/3.25])
                        if col == 0: 
                            copy_axes_properties(row,col,list_ax_[axes_index[2*row+col]], inset_ax, list_ax1[axes_index[2*row+col]],list_ax_[axes_index[2*row+col+1]])
                        else:
                            copy_axes_properties(row,col,list_ax_[axes_index[2*row+col]], inset_ax, list_ax1[axes_index[2*row+col]],list_ax_[axes_index[2*row+col-1]])

                                

                        #         inset_ax.imshow(ax2[(12*row)+col].get_images()[0].get_array().data)
                        #         if col == 0:
                        #             if row % 2 == 0: 
                        #                 inset_ax.text(25,10,ax2[0].get_ylabel(),color = "white",size=10)
                                    
                        #             else:
                        #                 inset_ax.text(25,10,ax2[12].get_ylabel(),color = "white", size=10)
                                
                        #         if row % 2 == 0 and col % 4 == 0: 
                        #                 inset_ax.text(10,10, label_marker_symbols[label_marker_symbols_counter], color = "white", size = 24)
                        #                 label_marker_symbols_counter+=1
                        #         elif row % 2 == 1 and (col + 1) % 4 == 0:
                        #             inset_ax.text(50,50, labels[label_counter], color = "white", size = 10)
                        #             label_counter+=1 
                            
                        
                        

                            
                            
                            
                            

                        # copy_axes_properties(i, axes[idx[0]], ax, axes[idx[2]], idx[1])
                        # #copy_axes_properties(axes[r_start], ax, axes[r_end], idx[1])
                        
                        # # for line in axes[idx[0]].get_lines():
                        # #         label = line.get_label() if line.get_label() != '_nolegend_' else None
                        # #         # Copying line properties like color, linestyle, marker, etc.
                                
                        # #         ax.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                        # #                     linestyle=line.get_linestyle(), marker=line.get_marker(), label=label) 
                                
                        
                        # # ax_twin = axes[idx[0]].twinx()
                        # # ax_twin.set_ylim(axes[idx[2]].get_ylim())
                        # # ax_twin.set_xlabel(axes[idx[0]].get_xlabel(),fontsize=14)
                        # # ax_twin.set_ylabel(axes[idx[0]].get_ylabel(),fontsize=14)
                        
                        # # for line in axes[idx[2]].get_lines():
                        # #         label = line.get_label() if line.get_label() != '_nolegend_' else None
                        # #         # Copying line properties like color, linestyle, marker, etc.
                                
                        # #         ax.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                        # #                     linestyle=line.get_linestyle(), marker=line.get_marker(), label=label) 
                        
                        # # set_sci_notation_label(
                        # #     axes[idx[2]], corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                        # #     textsize = 8,
                        # # )
                        
                        # # ax.set_xlabel(axes[idx[0]].get_xlabel(),fontsize=14)
                        # # ax.set_ylabel(axes[idx[0]].get_ylabel(),fontsize=14)
                        
                        # # ax.tick_params(axis='x',labelsize=10)
                        # # ax.tick_params(axis='y',labelsize=10)

                
                    
                        if row == 0:
                        
                            labelfigs(inset_ax,
                                    string_add="Best",
                                    loc ='tl',
                                    size=15,
                                    inset_fraction=(0.05,0.37),
                                    style = 'b',
                                    horizontalalignment = "center"
                                    )
                            
                        # get legend handles and their corresponding labels
                            handles1, labels1 = inset_ax.get_legend_handles_labels()
                            handles2, labels2 = list_ax1[axes_index[2*row+col]].get_legend_handles_labels()
                            

                            # zip labels as keys and handles as values into a dictionary, ...
                            # so only unique labels would be stored 
                            #dict_of_labels = dict(zip(labels, handles))

                            # use unique labels (dict_of_labels.keys()) to generate your legend
                            # ax.legend(dict_of_labels.values(), dict_of_labels.keys(),fontsize=10,
                            #         loc = 1)#, bbox_to_anchor = (0,0.50))
                            
                            inset_ax.legend(handles1 + handles2,labels1+labels2, loc=(0.45,0.55),fontsize = 10)
                            
                            # ax.set_xlabel("")
                            # ax.set_xticks([])

                        elif row == 1:
                            labelfigs(inset_ax,
                                    string_add="Median",
                                    loc ='tl',
                                    size=15,
                                    inset_fraction=(0.05,0.37),
                                    style = 'b',
                                    horizontalalignment = "center"
                                    )
                            #ax.set_xlabel("")
                            #ax.set_xticks([])
                        elif row == 2:
                            labelfigs(inset_ax,
                                    string_add="Worst",
                                    loc ='tl',
                                    size=15,
                                    inset_fraction=(0.05,0.37),
                                    style = 'b',
                                    horizontalalignment="center"
                                    )
                            ax.set_xticks([1.2,1.3,1.4])
                            
                        
                        labelfigs(inset_ax,
                            number=2*row+col,
                            loc ='tl',
                            size=15,
                            inset_fraction=(0.05,0.19),
                            style = 'b'
                            )


# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()

BE_viz.Printer.savefig(
                ax.figure, "Figure_3"
            )

## second part of figure

# LSQF_ = {'resampled': True,
#                 'raw_format': 'complex',
#                 'fitter': 'LSQF',
#                 'scaled': False, #True
#                 'output_shape': 'index', #pixels
#                 'measurement_state': 'all',
#                 'resampled_bins': 165,
#                 'LSQF_phase_shift': 1.5707963267948966, #None
#                 'NN_phase_shift': None,
#                 'noise': 0}

# LSQF_Params = dataset.SHO_fit_results(state = LSQF_)


# set_seeds(seed=42)

# postprocessor = ComplexPostProcessor(dataset)


# model_ = Multiscale1DFitter(SHO_fit_func_nn, # function 
#                             dataset.frequency_bin, # x data
#                             2, # input channels
#                             4, # output channels
#                             dataset.SHO_scaler, 
#                             postprocessor)




# # instantiate the model
# model = Model(model_, dataset, training=False, model_basename="SHO_Fitter_original_data")

# model.load(
#     "./Trained Models/SHO Fitter/2024-09-23_14-36-21_nn_benchmarks_noise/SHO_Fitter_model_optimizer_Adam_epoch_4_train_loss_0.040321211942850994.pth"
# )

# #model.load_state_dict(torch.load("./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_599_train_loss_0.005903387442231178.pth"))

# X_data, Y_data = dataset.NN_data()

# # you can view the test and training dataset by replacing X_data with X_test or X_train
# pred_data, scaled_param, parm = model.predict(X_data)

# fig = BE_viz.SHO_switching_maps_test([parm,LSQF_Params],labels = ["NN","LSQF"])



# # Set the figure size with 24 inches width
# fig.set_size_inches(24, 10, forward=True)

# # Show the layout
# fig.show()



In [ ]:
np.linspace(-6,6,7)

In [ ]:
inset_ax.images[0]

In [ ]:
cbar.ax.get_images.__sizeof__()

In [ ]:
print(plt.gcf().get_size_inches(), plt.gcf().get_dpi())


In [ ]:
cbar.ax.figure.dpi

In [ ]:
cbar.ax.figure.get_size_inches()

In [ ]:
ax.figure.dpi

In [ ]:
ax.figure.dpi_scale_trans.inverted()

In [ ]:
print(ax.figure.get_window_extent().transformed(ax.figure.dpi_scale_trans.inverted()).width)
print(ax.figure.get_window_extent().transformed(ax.figure.dpi_scale_trans.inverted()).height)

In [ ]:
ax.figure.get_size_inches()*ax.figure.dpi

In [ ]:
np.array([2**16,2**16]-ax.figure.get_size_inches()*ax.figure.dpi)

In [ ]:
cbar.formatter.get_offset()

In [ ]:
9e-5+1e-5 == 1e-4

In [ ]:
exponent_axis

In [ ]:
from matplotlib.ticker import EngFormatter


In [ ]:
fmt = ScalarFormatter(useMathText=True)
cbar.formatter = fmt
cbar.formatter.set_format("1.2e")
cbar.update_ticks()


In [ ]:
print(np.linspace(np.min(clims[0][int(np.floor(0 % 4))]),
            np.max(clims[0][int(np.floor(0 % 4))]),5))
print("*"*20)
print(np.linspace(np.min(clims[0][int(np.floor(1 % 4))]),
            np.max(clims[0][int(np.floor(1 % 4))]),5))
print("*"*20)
print(np.linspace(np.min(clims[0][int(np.floor(2 % 4))]),
            np.max(clims[0][int(np.floor(2 % 4))]),5))
print("*"*20)
print(np.linspace(-3,3,7))

In [ ]:
clims

In [ ]:
print(np.linspace(np.min(clims[0][int(np.floor(2 % 4))]),
            np.max(clims[0][int(np.floor(2 % 4))]),5))

In [ ]:
clims[0][int(np.floor(2 % 4))][1]

In [ ]:
print(np.floor(np.log10(clims[0][int(np.floor(0 % 4))][1])).astype(int))
print("*"*20)
print(np.floor(np.log10(clims[0][int(np.floor(1 % 4))][1])).astype(int))
print("*"*20)
print(np.floor(np.log10(np.absolute(clims[0][int(np.floor(2 % 4))][1]))).astype(int))
print("*"*20)


In [ ]:
np.floor(np.log10(abs(clims[0][int(np.floor(2 % 4))][1]))).astype(int)


In [ ]:
np.log10(160)

In [ ]:
format(1320000,"1.2e")

In [ ]:
fmt.format = '1.2e'


In [ ]:
fmt = EngFormatter(useMathText=True,places=2)
fmt.set_powerlimits((0, 0))
#fmt.places("%1.2f")

In [ ]:
clims=[
                    (0, 1.4e-4),  # amplitude
                    (1.31e6, 1.33e6),  # resonance frequency
                    (-230, -160),  # quality factor
                    (-np.pi, np.pi),  # phase
                ],
clims[0]

In [ ]:
clims[0][int(np.floor(col % 4))]

In [ ]:
cbar = plt.colorbar(inset_ax.images[0],location = 'bottom', cax = bar_ax[0], format = fmt)
            
#cbar.ax.set_ylim(clims[0][int(np.floor(col % 4))])

ax.inset_axes.cbar(clims[0][int(np.floor(col % 4))])                       

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import ScalarFormatter

# Sample data
data = np.random.rand(5, 5)

# Create a figure and axis
fig, ax = plt.subplots()
im = ax.imshow(data)

# Add a colorbar
fmt = ScalarFormatter(useMathText=True)
fmt.set_powerlimits((0, 0))

# Add a colorbar
cbar = plt.colorbar(im, ax=ax, location='bottom', format=fmt)

#cbar.ax.yaxis.set_offset_position('left')  # Move exponent to the right side
# Customize the colorbar's formatter
#cbar.formatter.set_scientific(True)
#cbar.formatter.set_powerlimits((0,0))
#cbar.update_ticks()
print("exponent:",cbar.formatter.get_offset())
cbar.ax.text(0,0,cbar.formatter.get_offset(),size=20)
exponent = -3
cbar.ax.text(1.05,0,f'$\\times 10^{{exponent}}$',size=20)
cbar.ax.text(0.05,0,f'$\\times 10^{{{exponent}}}$',size=20)
#cbar.ax.text(0.25,0,f'$\\times 10^{exponent}$',size=20)


cbar.ax.text(0.5,0,10,size=20)
cbar.ax.xaxis.offsetText.set_visible(False)
print("ex:",cbar.ax.xaxis.get_offset_text())

# # Move the exponent to the right of the colorbar
# if cbar.formatter.get_offset():
#     cbar.ax.xaxis.offsetText.set_visible(False)

#     cbar_label = ax.text(
#         5, 0.5,  # Position to the right of the colorbar
#         cbar.formatter.get_offset(),  # Exponent text
#         transform=cbar.ax.transAxes,
#         ha='left',
#         va='center'
#     )
# #     #cbar.formatter.set_useOffset(False)  # Remove default offset display
# #     #cbar.ax.xaxis.offsetText.set_visible(False)  # Hide the default exponent

plt.show()


In [ ]:
cbar.formatter.get_offset()

In [ ]:
cbar.formatter.get_offset()

In [ ]:
data

In [ ]:
for i in range(6):
    print(i, list_ax_[i].get_label())

In [ ]:
for col in range(12):
    print(np.floor(col % 4))


In [ ]:
ax1.get_legend_handles_labels()

In [ ]:
ax_.get_legend_handles_labels()

In [ ]:
max(list_ax_[2].get_ylim())

In [ ]:
print(6e-3,1e-2)

In [ ]:
list_ax_

In [ ]:
ax_SHO_comp[0]

In [ ]:
idx

In [ ]:
ax_SHO_comp

In [ ]:
for row in range(3):
    for col in range(2):
        print(2*row+col, list_ax_[2*row+col])

In [ ]:
for row in range(3):
    for col in range(2):
        print(row, col, 2*row+col)

In [ ]:
for col in range(2):
    for row in range(3):
        print(row,col, 2*row+col)

In [ ]:
for i in range(6):
    print(i, list_ax_[i].get_ylim())

In [ ]:
for line in list_ax_[5].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        plt.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
 
 
 
# list_ax_[1].set_ylim(list_ax_[1].get_ylim())        
# ax_twin = list_ax_[1].twinx()
# ax_twin.set_ylim(list_ax1[1].get_ylim())

# for line in list_ax1[1].get_lines():
#         label = line.get_label() if line.get_label() != '_nolegend_' else None
#         # Copying line properties like color, linestyle, marker, etc.
#         plt.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
#                        linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
        

In [ ]:
[0,3]

In [ ]:
for line in ax_SHO_comp[0].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        plt.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
        
ax_twin = ax_SHO_comp[0].twinx()
ax_twin.set_ylim(ax_SHO_comp[1].get_ylim())
#ax_twin.set_yticks([-3,-2,-1,0,1,2,3])
#ax_twin.set_yticks([-np.pi,0,np.pi],labels =["$-\pi$","0","$\pi$"])

# if i % 2 == 0: 
#     ax_twin.set_ylabel("")
#     ax_twin.set_yticks([])
# else:
#     target_ax.set_ylabel("")
#     ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize = 25)

ax_twin.set_ylabel(ax_SHO_comp[1].get_ylabel(),fontsize = 25)


for line in ax_SHO_comp[1].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties for the twin axis
        ax_twin.plot(line.get_xdata()/1e6, line.get_ydata(), color=line.get_color(),
                        linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
        
plt.show()


In [ ]:
ax_twin

In [ ]:
gs[30:80,0:60]

# MAKE SURE THE PLOTS (1-6) ARE SQUARE AND THE X-AXIS HAS THE SAME LIMITS AND THEY PLOT THE CORRECT STUFF. 
# THINK ABOUT THE LABEL AND IF IT CLEARLY SHOWS EVERYTHING 

In [ ]:
gs[2].colspan

In [ ]:
np.linspace(0,96,13)

In [ ]:
np.linspace(0,100,11)

In [ ]:
ax.figure

In [ ]:
for col in range(12):
    print(np.floor(col/4))


In [ ]:
for row in range(6):
    for col in range(12):
        print((12*row)+col)

In [ ]:
print("\u25CF", "\u25BC", "\u25B2", "\u25BA", "\u25C0", "\u25A0","\u271A", "\u25C6","\u2605" )

In [ ]:
"\u25BC"

In [ ]:
-0.02+1/12

In [ ]:
for i in range(15):
    print(i, np.array(ax2[i].get_position()).flatten())


In [ ]:
ax2[0].figure.axes

In [ ]:
type(ax2[0].get_images()[0].get_array().data)

In [ ]:
ax2[0].get_ylabel()

In [ ]:
for row in range(6):
    for col in range(12):
        if row % 2 == 0 and col % 4 == 0: 
            print((12*row)+col , row, col)

In [ ]:
col+=1
col

In [ ]:
plt.imshow(ax2[0].get_images()[0].get_array().data,interpolation=None) #,label = ax2[0].get_ylabel())
plt.text(30,10,ax2[0].get_ylabel(),color = "white",size=48)
plt.text(10,10, '\u25CF', color = "white",size = 48)
#plt.axis("off")

In [ ]:
# Define a grid layout for the images
fig2= plt.figure(figsize=(8, 6))

# Add each image to a specific grid cell
ax1.add_subplot(6,12,1)
ax1.imshow(ax2[0].get_images()[0].get_array().data, cmap='viridis')
ax1.axis('off')
ax.add_child
# ax1 = fig2.add_subplot(3,3,2)
# ax1.imshow(ax2[0].get_images()[0].get_array().data, cmap='viridis')
# ax1.axis('off')

# ax1 = fig2.add_subplot(3,3,5)
# ax1.imshow(ax2[0].get_images()[0].get_array().data, cmap='viridis')
# ax1.axis('off')


plt.show()

In [ ]:
ax2[1].findobj()[0]

In [ ]:
type(ax2[0].figure)

In [ ]:
ax.add_image(ax2[0].figure)

In [ ]:
plt.imshow(ax2[1].axis())

In [ ]:
for line in ax2[0].get_lines():
    plt.plot(line.get_xdata(),line.get_ydata())
    print("xdata", line.get_xdata())
    
plt.show()

In [ ]:
ax.figure

In [ ]:
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas


# Render the figure as an image
canvas = FigureCanvas(ax2[0].figure)
canvas.draw()  # Render the figure onto the canvas
bbox = ax2[0].figure.get_tightbbox(canvas.get_renderer())#.transformed(ax2[0].figure.dpi_scale_trans.inverted())

#canvas.draw()  # Render the figure onto the canvas

width, height = ax2[0].figure.get_size_inches() * ax2[0].figure.dpi  # Original figure size in pixels


image = np.frombuffer(canvas.tostring_rgb(), dtype='uint8')
image = image.reshape(int(height), int(width), 3)  # Reshape to height x width x RGB

#image = image.reshape(canvas.get_width_height()[::-1] + (3,))

# Crop the image to the bounding box
bbox_pixels = (
    int(bbox.x0 * ax2[0].figure.dpi),
    int(bbox.y0 * ax2[0].figure.dpi),
    int(bbox.x1 * ax2[0].figure.dpi),
    int(bbox.y1 * ax2[0].figure.dpi),
)
image_cropped = image[bbox_pixels[1]:bbox_pixels[3], bbox_pixels[0]:bbox_pixels[2], :]

plt.imshow(image_cropped)
plt.axis('off')  # Hide the axis if desired


In [ ]:
ax2[71].figure

In [ ]:
bbox_pixels

In [ ]:
image_cropped.shape

In [ ]:
type(ax2[0:-2])

In [ ]:
image[:, bbox_pixels[0]:bbox_pixels[2], :].shape

In [ ]:
image.shape

In [ ]:
ax2[0].figure

In [ ]:
type(ax2[0].figure)

In [ ]:
plt.imshow(ax2[0].figure)

In [ ]:
ax2[1].figure

In [ ]:
plt.plot(ax2[1].axis())

In [ ]:
clims[j]

In [ ]:
ax2[axis_start + j].images[0].set_clim(clims[j])


In [ ]:
print(parm.shape, LSQF_Params.shape)

In [ ]:
dataset.dataset = "Raw_Data"

In [ ]:
dataset.measurement_state = 'all'

In [ ]:
for line in fig.axes[1].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
 
        plt.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
plt.show()

In [ ]:
len(dataset.dc_voltage)

In [ ]:
len(dataset.get_cycle(dataset.dc_voltage))

In [ ]:
hyst,bias = dataset.get_hysteresis(plotting_values = True)

In [ ]:
bias.shape

In [ ]:
hyst.shape

In [ ]:
plt.plot(dataset.roll_hysteresis(bias))

In [ ]:
dataset.num_cycles

In [ ]:
dataset.get_voltage.shape

In [ ]:
dataset.cycle = 2

In [ ]:
plt.scatter(np.linspace(0,10,10),dataset.dc_voltage[0:10])


In [ ]:
plt.plot(dataset.get_voltage,color = 'blue')
plt.plot(dataset.dc_voltage,color = 'orange')
plt.plot(dataset.get_cycle(dataset.dc_voltage),color = 'black')

In [ ]:
plt

## This might be part of the problem with the voltages. `num_cycles == 4` and `dc_voltage` has 192 steps so it results in only 48 total steps, not the expected 96

In [ ]:
BE_viz.SHO_switching_maps(parm,
                          clims=[
                                    (0, 1.4e-4),  # amplitude
                                    (1.31e6, 1.33e6),  # resonance frequency
                                    (-230, -160),  # quality factor
                                    (0, 2*np.pi),  # phase
                                ], 
                          cycle=3,
                          
                          
                          
                          )

In [ ]:
BE_viz.SHO_switching_maps(parm,
                          clims=[
                                    (0, 1.4e-4),  # amplitude
                                    (1.31e6, 1.33e6),  # resonance frequency
                                    (-230, -160),  # quality factor
                                    (0, 2*np.pi),  # phase
                                ], 
                          cycle=3,
                          
                          
                          
                          );

Go from source code. Run the SHO_fit_results

Figure out how to do it for the voltage points in the figure. Use the `state` keyword?

state = {"fitter": "LSQF", "resampled": True, "scaled": True, "label": "Scaled", "noise": noise}

LSQF_ = {'resampled': True,
                'raw_format': 'complex',
                'fitter': 'LSQF',
                'scaled': True,
                'output_shape': 'index',
                'measurement_state': 'all',
                'resampled_bins': 165,
                'LSQF_phase_shift': 1.5707963267948966,
                'NN_phase_shift': None,
                'noise': noise}


In [ ]:
true_state

In [ ]:
out_state

In [ ]:
test = dataset.SHO_fit_results(state = out_state, model = model)

In [ ]:
test.shape

In [ ]:
test[:,0].reshape(60,60,96,-1).shape

In [ ]:
plt.imshow(test[:,0].reshape(60,60,96,-1)[:,:,0,0])

In [ ]:
idx[0]

In [ ]:
len(axes)

Figure 3. SHO fitting results of DNN in comparison with LSQF method’s results. a,c,e Best, median and worst predictions of LSQF method. b,d,f Best, median and worst predictions of neural network trained with ADAHESSIAN. g Distributions of predicted parameters. h Band-excitation waveform with switching voltage comparing maps of four parameters (i-q) predicted by NN and LSQF

## Figure 4

In [ ]:
h5_loop_fit, h5_loop_group = dataset.LSQF_Loop_Fit()

In [ ]:
%load_ext autoreload
%autoreload 2

from belearn.dataset.dataset import BE_Dataset
from belearn.viz.viz import Viz
from m3util.viz.printing import printer
printing = printer(basepath = './Figures/')


In [ ]:
# instantiate the visualization object
image_scalebar = [2000, 500, "nm", "br"]

In [ ]:
# instantiate the visualization object
image_scalebar = [2000, 500, "nm", "br"]


# Specify the filename and the path to save the file
filename = "data_raw.h5"
save_path = "./Data"

data_path = save_path + "/" + filename

# instantiate the dataset object
dataset = BE_Dataset(data_path)


In [ ]:

BE_viz = Viz(dataset, printing, verbose=True, 
             SHO_ranges = [(0,1.5e-4), (1.31e6, 1.33e6), (-300, 300), (-np.pi, np.pi)], 
             image_scalebar = image_scalebar)

In [ ]:

from autophyslearn.spectroscopic.nn import Multiscale1DFitter, Model
from autophyslearn.postprocessing.complex import ComplexPostProcessor

In [ ]:
dataset.SHO_Scaler.var_

In [ ]:
dataset.loop_param_scaler

In [ ]:
model_.scaler.fit(voltage)

In [ ]:
model_.scaler.var_

In [ ]:
model_.post_processing

In [ ]:
dataset.hysteresis_scaler.std

In [ ]:
device='cuda:1'

In [ ]:
postprocessor = ComplexPostProcessor(dataset,device=device)
print(postprocessor.std)


In [ ]:
#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn


datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"
device = 'cuda:1'

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
            #BE_viz.loop_fitting_function_torch, # function 
                hysteresis_nn,  # function

                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/jca92/Rapid-Fitting-BEPFM-NN/notebooks/7_Hysteresis_Fitter.ipynb",
                device=device)

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 100}


train =  True

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 500,
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.00602861393450035.pth"
    )

In [ ]:
?? hysteresis_nn

In [ ]:
#from m3_learning.be.loop_fitter import loop_fitting_function_torch
from m3util.ml.optimizers.TrustRegion import TRCG
import torch.optim as optim
from belearn.functions.hysteresis import hysteresis_nn


datafed_path = "2024_SHO_Fitting/Original_NN_SHO_Fitter"
device = 'cuda:1'

data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
# V = np.swapaxes(np.atleast_2d(dataset.get_voltage), 0, 1).astype(np.float64)


model_ = Multiscale1DFitter(
            #BE_viz.loop_fitting_function_torch, # function 
                hysteresis_nn,  # function

                            voltage[:,0].squeeze(), # x data
#                             V.squeeze(),
                            1, # input channels
                            9, # output parameters
                            dataset.loop_param_scaler,
                            loops_scaler=dataset.hysteresis_scaler,
                            device=device
                            )

# instantiate the model
model = Model(model_, dataset, training=True, model_basename="SHO_Fitter_original_data",
                datafed_path=datafed_path,
                script_path="/home/jca92/Rapid-Fitting-BEPFM-NN/notebooks/7_Hysteresis_Fitter.ipynb",
                device=device)

from sklearn.model_selection import train_test_split


X_train, X_test = train_test_split(data.reshape(-1,96), test_size=0.2, random_state=42, shuffle=True)

X_train = np.atleast_3d(X_train)

optimizer = {
    "name": "TRCG", 
    "optimizer": TRCG,
    "closure_size": 1,
    "cgopttol": 1e-3,
    "c0tr": 0.2,
    "c1tr": 0.25,
    "c2tr": 0.75,
    "t1tr": 0.75,
    "t2tr": 2.0,
    "radius_max": 5.0,  
    "radius_initial": 1.0,
    "radius" : 1.0,
    "device": device,
    "ADAM_epochs": 100}


train =  True

if train:
    # fits the model
    model.fit(
        X_train,
        1024,
        optimizer=optimizer,
        epochs = 500,
    )
else:
    model.load(
       # "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_epoch_5_train_loss_0.0449272525189978.pth"
       "./Trained Models/SHO Fitter/SHO_Fitter_original_data_model_optimizer_Trust Region CG_epoch_499_train_loss_0.00602861393450035.pth"
    )

In [ ]:
X_train.device

In [ ]:
import sklearn
print(sklearn.__version__)


In [ ]:
from datafed.CommandLib import API

df_api = API()

df_api.getAuthUser()

In [ ]:
from m3util.viz.style import set_style

set_style("printing")


In [ ]:
n = 1

data = ("LSQF", "NN")

fig = BE_viz.hysteresis_comparison(
    data,
    nn_model=model,
    filename="Figure_XX_LSQF_NN_bmw_comparison_test",
)

In [ ]:
n = 1

data = ("LSQF", "NN")

fig = BE_viz.hysteresis_comparison(
    data,
    nn_model=model,
    filename="Figure_XX_LSQF_NN_bmw_comparison_test",
)

In [ ]:
fig

In [ ]:
axes = fig.axes

In [ ]:
axes

In [ ]:
import torch

In [ ]:
data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)

fig = BE_viz.violin_plot_comparison_hysteresis(model,
                                         torch.atleast_3d(torch.tensor(data.reshape(-1, 96))),
                                         filename="Figure_XX_Violin") 

In [ ]:
fig.axes

In [ ]:
axes.extend(fig.axes)

In [ ]:
data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

pred_recon, pred_params_scaled, pred_params = model.predict(
    data,
    1024,
    translate_params=False,
    is_SHO=False
)

fig = BE_viz.hysteresis_maps(pred_params, cycle=0, filename="Figure_XX_NN_Hysteresis_Maps_test")

In [ ]:
plt.rcParams["xtick.labelsize"]

In [ ]:
axes

In [ ]:
axes

In [ ]:
for line0 in axes[0].get_lines():
    x_data_0 = line0.get_xdata()
    y_data_0 = line0.get_ydata()

for line1 in axes[1].get_lines():
    x_data_1 = line1.get_xdata()
    y_data_1 = line1.get_ydata()

print(min(x_data_0==x_data_1),min(y_data_0==y_data_1) )



In [ ]:
y_data_0

In [ ]:
y_data_0

In [ ]:
y_data_1

In [ ]:
for line in axes[3].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        
        plt.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
plt.show()

In [ ]:
for line in axes[3].get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        
        plt.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
plt.show()

In [ ]:
np.linspace(np.min(line.get_xdata()),np.max(line.get_xdata()),9)

In [ ]:
[0,1,2,3,4,5,6][-7]

In [ ]:
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    print(i,-i-2)

In [ ]:
ax.get_h

In [ ]:
import pandas as pd
import seaborn as sns
import itertools
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front



def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    target_ax.set_xlabel(source_ax.get_xlabel(),fontsize=14)
    target_ax.set_ylabel(source_ax.get_ylabel(),fontsize=14)
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Handle twin axes if present
    if secondary_ax:
        ax_twin = target_ax.twinx()
        ax_twin.set_ylim(secondary_ax.get_ylim())
        ax_twin.set_ylabel(secondary_ax.get_ylabel(),fontsize=14)

        for line in secondary_ax.get_lines():
            label = line.get_label() if line.get_label() != '_nolegend_' else None
            # Copying line properties for the twin axis
            ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                         linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Copy legends
    if source_ax.get_legend():
        target_ax.legend(fontsize='large')

    if secondary_ax and secondary_ax.get_legend():
        ax_twin.legend(fontsize='large')
        
    #BE_viz._scientific_notation_dual(target_ax,ax_twin)
    
    
    
    set_sci_notation_label(
                target_ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                textsize = 8,
            )
    
    # set_sci_notation_label(
    #             ax_twin, corner="top right", axis="y", stroke_color="w", linewidth=0.5,
    #             textsize=8,
    #         )
    
    target_ax.tick_params(axis='x',labelsize=10)
    target_ax.tick_params(axis='y',labelsize=10)

    # ax_twin.tick_params(axis='x',labelsize=10)
    # ax_twin.tick_params(axis='y',labelsize=10)

    target_ax.set_xticks(np.linspace(np.min(line.get_xdata()),np.max(line.get_xdata()),9))

# Create a figure
fig = plt.figure(figsize=(16, 16))

# Define the GridSpec layout
gs = GridSpec(6, 4, figure=fig)

order = [[0, 'twin', 6],
         [1, 'twin', 7],
         [2, 'twin', 8],
         [3, 'twin', 9],
         [4, 'twin', 10],
         [5, 'twin', 11],
         ['violin'],
        ]

# List of axes indices in GridSpec for each subplot
subplot_specs = [(0, 1, 0, 1), # a
                 (0, 1, 1, 2), # b
                 (1, 2, 0, 1), # c
                 (1, 2, 1, 2), # d
                 (2, 3, 0, 1), # e
                 (2, 3, 1, 2), # f
                 (0, 3, 2, 6), # g
                ]

# Create and set up subplots
for i, (r_start, r_end, c_start, c_end) in enumerate(subplot_specs):
    ax = fig.add_subplot(gs[r_start:r_end, c_start:c_end])
    if i < len(axes):
        idx = order[i]
        if idx[0] == 'violin':
            df = pd.DataFrame()

            # uses the model to get the predictions
            pred_data, scaled_param, params = model.predict(data, is_SHO=False)

            true = dataset.LSQF_hysteresis_params().reshape(-1, 9)

            true_scaled = dataset.loop_param_scaler.transform(true)

            # Builds the dataframe for the violin plot
            true_df = pd.DataFrame(
                true, columns=["a0", "a1", "a2", "a3", "a4",
                               "b0", "b1", "b2", "b3"]
            )
            predicted_df = pd.DataFrame(
                scaled_param, columns=["a0", "a1", "a2", "a3", "a4",
                                       "b0", "b1", "b2", "b3"]
            )

            # merges the two dataframes
            df = pd.concat((predicted_df, true_df))

            # adds the labels to the dataframe
            names = [true_scaled, scaled_param]
            names_str = ["NN", "LSQF"]
            labels = ["a0", "a1", "a2", "a3", "a4", "b0", "b1", "b2", "b3"]

            # adds the labels to the dataframe
            for j, name in enumerate(names):
                for i, label in enumerate(labels):
                    dict_ = {
                        "value": name[:, i],
                        "parameter": np.repeat(label, name.shape[0]),
                        "dataset": np.repeat(names_str[j], name.shape[0]),
                    }

                    df = pd.concat((df, pd.DataFrame(dict_)))
                    
            
                    

#             # builds the plot
#             fig, ax = plt.subplots(figsize=(4, 4))

              # Reset index to handle potential duplicated columns or indices
            df = df.reset_index(drop=False)
            
            # plots the data
            sns.violinplot(
                data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax
            )

            # labels the figure and does some styling
            labelfigs(ax, string_add = 'g', loc ='tl',size=20, style="b", inset_fraction=(0.05,0.12))
            ax.set_ylabel("Scaled Hysteresis Results",fontsize=16)
            ax.set_xlabel("")
            
            ax.tick_params(axis='x',labelsize=14)
            ax.tick_params(axis='y',labelsize=14)

            # Get the legend associated with the plot
            legend = ax.get_legend()
            legend.set_title("")
            plt.setp(legend.get_texts(), fontsize='large') # Set the label size
        else:
           # copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
            #copy_axes_properties(axes[r_start], ax, axes[r_end], idx[1])
            #copy_axes_properties(axes[r_start], ax)
            for line in axes[-i-2].get_lines():
                    label = line.get_label() if line.get_label() != '_nolegend_' else None
                    # Copying line properties like color, linestyle, marker, etc.
                    
                    ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                                linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)
                    
                   # ax.legend(fontsize='large')
            

            set_sci_notation_label(
                ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5,
                textsize = 8,
            )
            
            
            ax.tick_params(axis='x',labelsize=10)
            ax.tick_params(axis='y',labelsize=10)

            # ax_twin.tick_params(axis='x',labelsize=10)
            # ax_twin.tick_params(axis='y',labelsize=10)

            ax.set_xticks(np.linspace(np.min(line.get_xdata()),np.max(line.get_xdata()),9))
            
            if i in [0,1]:
              
                labelfigs(ax,
                        string_add="Best",
                        loc ='tl',
                        size=8,
                        inset_fraction=(0.04,0.5),
                        style = 'b'
                        )
                
               # get legend handles and their corresponding labels
                handles, labels = ax.get_legend_handles_labels()

                # zip labels as keys and handles as values into a dictionary, ...
                # so only unique labels would be stored 
                dict_of_labels = dict(zip(labels, handles))

                # use unique labels (dict_of_labels.keys()) to generate your legend
                ax.legend(dict_of_labels.values(), dict_of_labels.keys(),fontsize='large',
                          loc = (0.02,0.65))#, bbox_to_anchor = (0,0.50))

            elif i in [2,3]:
                labelfigs(ax,
                        string_add="Median",
                        loc ='tl',
                        size=8,
                        inset_fraction=(0.04,0.5),
                        style = 'b'
                        )
                
            elif i in [4,5]:
                labelfigs(ax,
                        string_add="Worst",
                        loc ='tl',
                        size=8,
                        inset_fraction=(0.04,0.5),
                        style = 'b'
                        )
  
            
            labelfigs(ax,
                number=i,
                loc ='tl',
                size=12,
                inset_fraction=(0.05,0.15),
                style = 'b'
                )

            # if i == 0:
            #     ax.set_title("Least Square Fit")
            # elif i==1:
            #     ax.set_title("NN with Trust Region CG")
            
            # set_sci_notation_label(
            #     ax, corner="top left", axis="y", stroke_color="w", linewidth=0.5
            # )
            
            # set_sci_notation_label(
            #     ax, corner="top right", axis="y", stroke_color="w", linewidth=0.5
            # )
            

            

# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()


filename = "Figure_4a_test"

BE_viz.Printer.savefig(
                fig, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        


data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

pred_recon, pred_params_scaled, pred_params = model.predict(
    data,
    1024,
    translate_params=False,
    is_SHO=False
)
fig = BE_viz.hysteresis_maps(pred_params, cycle=0);

# Set the figure size with 24 inches width
fig.set_size_inches(24, 10, forward=True)


filename = "Figure_4b_test"


# for i in range(len(fig.axes)):
#     if fig.axes[i].get_xlabel() == "a0":
#         label_index = i
        


labelfigs(fig.axes[0],
        string_add="h",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )


labelfigs(fig.axes[9],
        string_add="i",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )

BE_viz.Printer.savefig(
                fig, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        



# Show the layout
fig.show()

In [ ]:
fig = BE_viz.hysteresis_maps(pred_params, cycle=0);

# Set the figure size with 24 inches width
fig.set_size_inches(24, 10, forward=True)


filename = "Figure_4b"


# for i in range(len(fig.axes)):
#     if fig.axes[i].get_xlabel() == "a0":
#         label_index = i
        


labelfigs(fig.axes[0],
        string_add="h",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )


labelfigs(fig.axes[9],
        string_add="i",
        loc ='tl',
        size=20,
        inset_fraction=(-0.2,0),
        style = 'b'
        )

BE_viz.Printer.savefig(
                fig, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        



# Show the layout
fig.show()

In [ ]:
for i in range(len(fig.axes)):
    if fig.axes[i].get_xlabel() == "a0":
        print(i)

In [ ]:
fig.axes[20].get_xlabel()

In [ ]:
"Axes" in np.array(fig.axes[0])

In [ ]:
np.where("a0" in fig.axes)

In [ ]:
fig.axes[np.where("a0" in fig.axes)[0]]

In [ ]:
import pandas as pd
import seaborn as sns
import itertools
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from m3util.viz.text import set_sci_notation_label, labelfigs, bring_text_to_front



def copy_axes_properties(source_ax, target_ax, secondary_ax=None, type_ax=None):
    """Copy properties and data from source_ax to target_ax."""
    # Copy basic properties
    target_ax.set_xlabel(source_ax.get_xlabel())
    target_ax.set_ylabel(source_ax.get_ylabel())
    target_ax.set_xlim(source_ax.get_xlim())
    target_ax.set_ylim(source_ax.get_ylim())
    target_ax.set_title(source_ax.get_title())

    # Copy lines from the primary axis
    for line in source_ax.get_lines():
        label = line.get_label() if line.get_label() != '_nolegend_' else None
        # Copying line properties like color, linestyle, marker, etc.
        target_ax.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                       linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Handle twin axes if present
    if secondary_ax:
        ax_twin = target_ax.twinx()
        ax_twin.set_ylim(secondary_ax.get_ylim())
        ax_twin.set_ylabel(secondary_ax.get_ylabel())

        for line in secondary_ax.get_lines():
            label = line.get_label() if line.get_label() != '_nolegend_' else None
            # Copying line properties for the twin axis
            ax_twin.plot(line.get_xdata(), line.get_ydata(), color=line.get_color(),
                         linestyle=line.get_linestyle(), marker=line.get_marker(), label=label)

    # Copy legends
    if source_ax.get_legend():
        target_ax.legend(fontsize='large')

    if secondary_ax and secondary_ax.get_legend():
        ax_twin.legend(fontsize='large')

# Create a figure
fig2 = plt.figure(figsize=(24, 24))

# Define the GridSpec layout
gs2 = GridSpec(10, 4, figure=fig2)

order2 = [[0, 'twin', 6],
         [1, 'twin', 7],
         [2, 'twin', 8],
         [3, 'twin', 9],
         [4, 'twin', 10],
         [5, 'twin', 11],
         ['violin'],
         ['hysteresis_maps']
        ]

# List of axes indices in GridSpec for each subplot
subplot_specs2 = [(0, 1, 0, 1), # a
                 (0, 1, 1, 2), # b
                 (1, 2, 0, 1), # c
                 (1, 2, 1, 2), # d
                 (2, 3, 0, 1), # e
                 (2, 3, 1, 2), # f
                 (0, 3, 2, 6),
                 (3, 6, 0, 5), # h & i together
                ]

# Create and set up subplots
for i2, (r_start2, r_end2, c_start2, c_end2) in enumerate(subplot_specs2):
        ax2 = fig2.add_subplot(gs2[r_start2:r_end2, c_start2:c_end2])
   # if i2 < len(axes):
        idx2 = order2[i2]
        if idx2[0] == 'violin':
            df = pd.DataFrame()

            # uses the model to get the predictions
            pred_data, scaled_param, params = model.predict(data, is_SHO=False)

            true = dataset.LSQF_hysteresis_params().reshape(-1, 9)

            true_scaled = dataset.loop_param_scaler.transform(true)

            # Builds the dataframe for the violin plot
            true_df = pd.DataFrame(
                true, columns=["a0", "a1", "a2", "a3", "a4",
                               "b0", "b1", "b2", "b3"]
            )
            predicted_df = pd.DataFrame(
                scaled_param, columns=["a0", "a1", "a2", "a3", "a4",
                                       "b0", "b1", "b2", "b3"]
            )

            # merges the two dataframes
            df = pd.concat((predicted_df, true_df))

            # adds the labels to the dataframe
            names = [true_scaled, scaled_param]
            names_str = ["NN", "LSQF"]
            labels = ["a0", "a1", "a2", "a3", "a4", "b0", "b1", "b2", "b3"]

            # adds the labels to the dataframe
            for j, name in enumerate(names):
                for i, label in enumerate(labels):
                    dict_ = {
                        "value": name[:, i],
                        "parameter": np.repeat(label, name.shape[0]),
                        "dataset": np.repeat(names_str[j], name.shape[0]),
                    }

                    df = pd.concat((df, pd.DataFrame(dict_)))
                    
            
                    

#             # builds the plot
#             fig, ax = plt.subplots(figsize=(4, 4))

              # Reset index to handle potential duplicated columns or indices
            df = df.reset_index(drop=False)
            
            # plots the data
            sns.violinplot(
                data=df, x="parameter", y="value", hue="dataset", split=True, ax=ax2
            )

            # labels the figure and does some styling
            labelfigs(ax2, 0, style="b")
            ax2.set_ylabel("Scaled Hysteresis Results")
            ax2.set_xlabel("")

            # Get the legend associated with the plot
            legend2 = ax2.get_legend()
            legend2.set_title("")
            plt.setp(legend2.get_texts(), fontsize='large') # Set the label size
            
            
        elif idx2[0] == "hysteresis_maps":
            data, voltage = dataset.get_hysteresis(scaled=True, loop_interpolated = True)
            data = torch.atleast_3d(torch.tensor(data.reshape(-1, 96)))

            pred_recon2, pred_params_scaled2, pred_params2 = model.predict(
                data,
                1024,
                translate_params=False,
                is_SHO=False
            )
            
            # something about figure 
            ax2.axes=BE_viz.hysteresis_maps(pred_params, cycle=0)[1]
  
            
        else:
           # copy_axes_properties(axes[idx[0]], ax, axes[idx[2]], idx[1])
           copy_axes_properties(axes[r_start2], ax2, axes[r_end2], idx2[1])
            

# Adjust the spacing between the plots as needed
plt.tight_layout()

# Show the layout
plt.show()



#fig = BE_viz.hysteresis_maps(pred_params, cycle=0);

# Set the figure size with 24 inches width
fig2.set_size_inches(24, 10, forward=True)


filename = "Figure_4_TEST"

BE_viz.Printer.savefig(
                fig2, filename, size=10, loc="tl", inset_fraction=(0.2, 0.2)
            )
        



# Show the layout
fig2.show()

In [ ]:
ax2.axis()

In [ ]:
fig2.

In [ ]:
BE_viz.hysteresis_maps(pred_params, cycle=0)[1].

In [ ]:
list(BE_viz.hysteresis_maps(pred_params, cycle=0)[1])

In [ ]:
ax2.axis = BE_viz.hysteresis_maps(pred_params, cycle=0)[1]

In [ ]:
ax2.set_figure

In [ ]:
ax2.axes.figure.axes.append(BE_viz.hysteresis_maps(pred_params, cycle=0)[1])

In [ ]:
type(BE_viz.hysteresis_maps(pred_params, cycle=0).axes[5])

In [ ]:
for i2, (r_start2, r_end2, c_start2, c_end2) in enumerate(subplot_specs2):
    print(i2,order2[i2],r_start2,r_end2,c_start2,c_end2)

In [ ]:
# Create a figure
fig2 = plt.figure(figsize=(24, 24))

# Define the GridSpec layout
gs2 = GridSpec(6, 4, figure=fig)


In [ ]:
df["parameter"]

In [ ]:
df["value"]

In [ ]:
df["value"]
0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
            ...   
287995   -0.601739
287996   -0.685297
287997   -0.603866
287998   -0.524238
287999   -0.646338
Name: value, Length: 288000, dtype: float64

In [ ]:
dataset.LSQF_hysteresis_params().shape
(60, 60, 4, 9)

In [ ]:
axes[idx[0]]

In [ ]:
len(axes)

Figure 4. Piezoelectric hysteresis loops fitting results of DNN in comparison with LSQF method’s results. a,c,e Best, median, and worst predictions of LSQF method. b,d,f Best, median, and worst predictions of neural network trained with Trust Region CG. g Distributions of predicted parameters. h Color maps of the signal of parameters resulted from LSQF method. i Color maps of the signal of parameters resulted from neural network.

## Figure 5

In [ ]:
fig, axs = plt.subplots(
            1,
            3,
            figsize=(12, 4),
            #gridspec_kw={"height_ratios": [1, 1]},
        )

